# HotBob Colab Runner

Use this notebook in Google Colab to run HotBob LLM-memory experiments on cloud GPU compute. It clones the GitHub repo, installs dependencies with `uv`, generates synthetic traces, trains a frozen-Qwen memory adapter, and runs evaluation.

Before running training, set **Runtime > Change runtime type > GPU**.

In [ ]:
import subprocess

print(sys.version)
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("No NVIDIA GPU visible. In Colab, choose Runtime > Change runtime type > GPU.")

## Configure the run

Set `REPO_URL` to your linked GitHub repository. `BRANCH` can be `master`, your current experiment branch, or any ref that exists on GitHub.

Generated datasets and checkpoints are intentionally written under `data/` and `runs/`, matching the repo's ignore policy.

In [ ]:
REPO_URL = "https://github.com/Popidge/hotbob.git"
BRANCH = "master"
PROJECT_DIR = "/content/hotbob"

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
SEED = 41

# Start small to confirm the notebook is healthy, then scale up.
TRAIN_N = 10_000
EVAL_N = 1_000
TRAIN_STEPS = 1_000
BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1

INTEGRATION_MODE = "prefix"  # prefix, attention_q, attention_o, attention_qo
MEMORY_STATE_MODE = "shared"  # shared, by_type
ATTENTION_PATCH_LAYERS = "last4"  # all, last4, or comma-separated layer indexes
CORRECTION_RANK = 16
WRITE_LOSS_WEIGHT = 0.2

TRAIN_TRACES = "data/llm_train_colab.jsonl"
EVAL_TRACES = "data/llm_eval_colab.jsonl"
CHECKPOINT = f"runs/qwen_memory/colab_{INTEGRATION_MODE}_{TRAIN_N}_{TRAIN_STEPS}.pt"

## Optional Hugging Face token

Public Qwen downloads usually work without a token. If you need one, add `HF_TOKEN` in Colab's Secrets panel, then run this cell.

In [ ]:
import os

try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("No HF_TOKEN secret found; continuing without one.")
except Exception:
    print("Colab userdata is unavailable; continuing without a Hugging Face token.")

## Clone or update the repo

In [ ]:
import os
import subprocess
from pathlib import Path

repo_path = Path(PROJECT_DIR)
if repo_path.exists():
    os.chdir(PROJECT_DIR)
    subprocess.run(["git", "fetch", "--all", "--prune"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    os.chdir(PROJECT_DIR)

subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=True)
subprocess.run(["git", "status", "--short", "--branch"], check=True)
print(f"Working directory: {Path.cwd()}")

## Install dependencies

This uses the repo's `uv.lock` and PyTorch CUDA index from `pyproject.toml`.

In [ ]:
!python -m pip install -q uv
!uv sync

## Quick verification

In [ ]:
import subprocess

import torch

print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))

subprocess.run(["uv", "run", "pytest"], check=True)

## Generate synthetic LLM traces

In [ ]:
!uv run python -m hotbob.llm.generate_data \
  --train-out {TRAIN_TRACES} \
  --eval-out {EVAL_TRACES} \
  --train-n {TRAIN_N} \
  --eval-n {EVAL_N} \
  --seed {SEED}

## Train a single memory adapter

In [ ]:
!uv run python -m hotbob.llm.train \
  --model {MODEL} \
  --traces {TRAIN_TRACES} \
  --steps {TRAIN_STEPS} \
  --batch-size {BATCH_SIZE} \
  --integration-mode {INTEGRATION_MODE} \
  --memory-state-mode {MEMORY_STATE_MODE} \
  --correction-rank {CORRECTION_RANK} \
  --attention-patch-layers {ATTENTION_PATCH_LAYERS} \
  --write-loss-weight {WRITE_LOSS_WEIGHT} \
  --freeze-base \
  --out {CHECKPOINT}

## Evaluate

In [ ]:
!uv run python -m hotbob.llm.evaluate \
  --checkpoint {CHECKPOINT} \
  --traces {EVAL_TRACES} \
  --model {MODEL} \
  --mode all \
  --batch-size {EVAL_BATCH_SIZE} \
  --decode-strategy score_answers

## Optional architecture comparison

Run this when you want matched prefix vs attention-correction variants from the same data and step count.

In [ ]:
import subprocess
import sys

RUN_ARCHITECTURE_COMPARISON = False
ARCHITECTURE_VARIANTS = [
    "prefix",
    "attention_q:last4",
    "attention_o:last4",
    "attention_qo:last4",
]

if RUN_ARCHITECTURE_COMPARISON:
    cmd = [
        "uv", "run", "python", "-m", "hotbob.llm.architecture_compare",
        "--model", MODEL,
        "--train-traces", TRAIN_TRACES,
        "--eval-traces", EVAL_TRACES,
        "--steps", str(TRAIN_STEPS),
        "--batch-size", str(BATCH_SIZE),
        "--eval-batch-size", str(EVAL_BATCH_SIZE),
        "--correction-rank", str(CORRECTION_RANK),
        "--write-loss-weight", str(WRITE_LOSS_WEIGHT),
        "--eval-modes", "teacher_forced",
    ]
    for variant in ARCHITECTURE_VARIANTS:
        cmd.extend(["--variant", variant])

    print("Running:", " ".join(cmd))
    result = subprocess.run(
        cmd,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if result.returncode:
        raise SystemExit(result.returncode)

## Inspect and download artifacts

Use this to grab checkpoints or reports before the Colab runtime is recycled.

In [ ]:
!find runs -maxdepth 4 -type f | sort

try:
    from google.colab import files

    print("To download the main checkpoint, run: files.download(CHECKPOINT)")
except Exception:
    pass